# DBI BHMM

While varying the diagonal operator and hence the rotation generator for each double-bracket iteration might be effective in the beginning of a diagonalization, it is unclear if varying it is necessary. In fact,Moore, Mahony and Helmke (MMH) discovered that a fixed $D$ may be sufficiently good for diagonalization when it's  non-degenerate (different eigenvalues) and in ascending order. What we do not know is whether it can be as effective as the variational DBI, and what fixed $D$ would be optimal for diagonalization.

Hence, we aim to understand how well BHMM performs, by following these steps:

1. Show that a fixed $D$ does diagonalize a hamiltonian (e.g. min-max, eigen). This one will work due to the BHMM theory. 

2. Compare some options for $D$, including constant, linear, and quadratic field vector $B$, within the constraint $D=H(B, J=0)=\sum B_iZ_i$, which is the magnetic field hamiltonian.

3. Extend the above to the classical Ising model, where $D=H(B,J)=\sum B_iZ_i + \sum J_{ij}Z_iZ_j$

# BHMM diagonalizes hamiltonian through DBI
In this section, our options include:

1. $D=\text{diag}(\min\Delta(H), \min\Delta(H)+\delta, ..., \max\Delta(H))$, where $\delta = \frac{\max\Delta(H)-\min\Delta(H)}{L}$.This corresponds to taking L equidistant values between the minimum and maximum of diagonal restriction of H. We refer to this as the "min-max" operator.
2. $D=(\lambda_1, ...,\lambda_{2^L})$, where $\lambda_i$ is the $i^{th}$ eigenvalue of $H$ in ascending order.While this choice may seem counterintuitive as having access to the eigenvalues trivializes the double-bracket flow algorithm, it is interesting to see its convergence rate.

## Initialize system
Our primary target hamiltonian is the transverse field Ising model. Here, we employ the `qibo` inbuilt hamiltonian, setting parameters `n` and `h`.

In [ ]:
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import math

from qibo import hamiltonians, set_backend
from qibo.hamiltonians import Hamiltonian, SymbolicHamiltonian
from qibo.quantum_info import random_hermitian
from qibo.models.dbi.double_bracket import DoubleBracketGeneratorType, DoubleBracketScheduling, DoubleBracketIteration
from qibo.models.dbi.utils import *

In [ ]:
def visualize_matrix(matrix, title=""):
    """Visualize hamiltonian in a heatmap form."""
    fig, ax = plt.subplots(figsize=(5,5))
    ax.set_title(title)
    try:
        im = ax.imshow(np.absolute(matrix), cmap="inferno")
    except TypeError:
        im = ax.imshow(np.absolute(matrix.get()), cmap="inferno")
    fig.colorbar(im, ax=ax)

In [ ]:
# set the qibo backend (we suggest qibojit if N >= 20)
set_backend("qibojit", "numba")

# hamiltonian parameters
nqubits = 5
h = 3

# define the hamiltonian
H = hamiltonians.TFIM(nqubits=nqubits, h=h)

# vosualize the matrix
visualize_matrix(H.matrix, title="Target hamiltonian")

## Generate the fixed diagonal operators

In [ ]:
# min-max
min_max = diagonal_min_max(H.matrix)
# eigen
eigen = np.diag(np.linalg.eigvalsh(H.matrix))
# compare the 2
diff = np.diag(min_max) - np.diag(eigen)


In [ ]:
plt.plot(np.diag(min_max), marker='o', label='min-max')
plt.plot(np.diag(eigen), marker='o', label='eigen')
plt.legend()
plt.title('Compare min-max and eigen operator')
plt.ylabel('Diagonal element')
plt.xlabel('Entry')

We see that both operators are ascending. While the `min-max` operator is non-degenerate, the `eigen` operator has some degeneracies. It is also noticeable that the `eigen` operator has a wider range of values compared to `min-max`. Next, we compare their performance in diagonalization.

In [ ]:
NSTEPS = 15
mode = DoubleBracketGeneratorType.single_commutator
scheduling = DoubleBracketScheduling.hyperopt
dbi_min_max = DoubleBracketIteration(deepcopy(H), mode=mode, scheduling=scheduling)
dbi_eigen = DoubleBracketIteration(deepcopy(H), mode=mode, scheduling=scheduling)
off_diagonal_min_max = [dbi_min_max.off_diagonal_norm]
off_diagonal_eigen = [dbi_eigen.off_diagonal_norm]
step_min_max_history = [0]
step_eigen_history = [0]

In [ ]:
for _ in range(NSTEPS):
    step_min_max = dbi_min_max.choose_step(d=min_max, step_min=1e-3, max_evals=300)
    step_eigen = dbi_eigen.choose_step(d=eigen, step_min=1e-3, max_evals=300)
    dbi_min_max(step=step_min_max, d=min_max)
    dbi_eigen(step=step_eigen, d=eigen)
    off_diagonal_min_max.append(dbi_min_max.off_diagonal_norm)
    off_diagonal_eigen.append(dbi_eigen.off_diagonal_norm)
    step_min_max_history.append(step_min_max)
    step_eigen_history.append(step_eigen)

In [ ]:
plt.plot(off_diagonal_min_max, marker='o', label='min-max')
plt.plot(off_diagonal_eigen, marker='o', label='eigen')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel(r'$|| \sigma(e^{sW}He^{-sW}) || $')
plt.title(f'DBI on TFIM n={nqubits}, h={h}')

In [ ]:
step_min_max_history

We notice that the `min-max` case stops diagonalizing after the very first step. This phenomenon is not changed regardless of the `hyperopt` settings.
We want to know if this is specific to TFIM. Hence, we test both on random Hamiltonians.

### Test on random hamiltonian

In [ ]:
seed = 5
H = Hamiltonian(nqubits=nqubits, matrix=random_hermitian(2**nqubits, seed=seed))
# min-max
min_max = diagonal_min_max(H.matrix)
# eigen
eigen = np.diag(np.linalg.eigvalsh(H.matrix))
# compare the 2
diff = np.diag(min_max) - np.diag(eigen)

In [ ]:
plt.plot(np.diag(min_max), marker='o', label='min-max')
plt.plot(np.diag(eigen), marker='o', label='eigen')
plt.legend()
plt.title('Compare min-max and eigen operator')
plt.ylabel('Diagonal element')
plt.xlabel('Entry')

In [ ]:
NSTEPS = 15
mode = DoubleBracketGeneratorType.single_commutator
scheduling = DoubleBracketScheduling.hyperopt
dbi_min_max = DoubleBracketIteration(deepcopy(H), mode=mode, scheduling=scheduling)
dbi_eigen = DoubleBracketIteration(deepcopy(H), mode=mode, scheduling=scheduling)
off_diagonal_min_max = [dbi_min_max.off_diagonal_norm]
off_diagonal_eigen = [dbi_eigen.off_diagonal_norm]
step_min_max_history = [0]
step_eigen_history = [0]

In [ ]:
for _ in range(NSTEPS):
    step_min_max = dbi_min_max.choose_step(d=min_max, step_min=1e-4,step_max=2)
    step_eigen = dbi_eigen.choose_step(d=eigen, step_min=1e-4,step_max=2)
    dbi_min_max(step=step_min_max, d=min_max)
    dbi_eigen(step=step_eigen, d=eigen)
    off_diagonal_min_max.append(dbi_min_max.off_diagonal_norm)
    off_diagonal_eigen.append(dbi_eigen.off_diagonal_norm)
    step_min_max_history.append(step_min_max)
    step_eigen_history.append(step_eigen)

In [ ]:
plt.plot(off_diagonal_min_max, marker='o', label='min-max')
plt.plot(off_diagonal_eigen, marker='o', label='eigen')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel(r'$|| \sigma(e^{sW}He^{-sW}) || $')
plt.title(f'Random Hamitlonian, seed={seed}')

The answer is Yes. That is, the `min-max` operator is only able to diagonalize `H_TFIM` for 1-step, but not necessarily for other hamiltonians.

In [ ]:
print(step_eigen_history)
print(step_min_max_history)

In [ ]:
plt.plot(step_min_max_history, label='min-max')
plt.plot(step_eigen_history, label='eigen')
plt.legend()
plt.xlabel('Iterations')
plt.ylabel('Step duration')

## Compare BHMM magnetic field operator
$D=\sum B_iZ_i$

In [ ]:
# generate Z operators
Z_ops = generate_onsite_Z_ops(nqubits)

In [ ]:
def B_order(nqubits, order, Z_ops):
    coef = np.array([(i+1)**order for i in range(nqubits)])
    return np.sum(coef.reshape(-1,1,1)*Z_ops, 0)

highest_order_test = 2
B_ops = [B_order(nqubits, order, Z_ops) for order in range(highest_order_test+1)]

In [ ]:
# visualizing the matrices: degeneracy
sort_B_ops = [np.sort(np.diag(B_op)) for B_op in B_ops]
for order, sort_B_op in enumerate(sort_B_ops):
    plt.plot(sort_B_op, label=f'order={order}')
plt.legend()
plt.title('Sorted diagonal entries')

In [ ]:
# visualize matrix
visualize_matrix(B_ops[0])

We see that with high order, the operator is less degenderate. We also see that the operators are not of ascending order. Hence, we would like to check whether having degeneracy and mixed order prohibits diagonalization.

In [ ]:
# initialize for tests
NSTEPS = 15
seed = 10
# H = Hamiltonian(nqubits=nqubits, matrix=random_hermitian(2**nqubits, seed=seed))

nqubits = 5
h = 2
H = hamiltonians.TFIM(nqubits=nqubits, h=h)

mode = DoubleBracketGeneratorType.single_commutator
scheduling = DoubleBracketScheduling.hyperopt
off_diag_norm_orders = []
step_orders = []
for order in range(highest_order_test+1):
    dbi = DoubleBracketIteration(deepcopy(H),mode=mode, scheduling=scheduling)
    off_diag_norm = [dbi.off_diagonal_norm]
    steps = [0]
    for _ in range(NSTEPS):
        step = dbi.choose_step(d=B_ops[order], max_evals=300)
        dbi(step=step, d=B_ops[order])
        off_diag_norm.append(dbi.off_diagonal_norm)
        steps.append(step)
    off_diag_norm_orders.append(off_diag_norm)
    step_orders.append(steps) 
        

In [ ]:
for order in range(highest_order_test+1):
    plt.plot(off_diag_norm_orders[order], label=f'order={order}')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel(r'$|| \sigma(e^{sW}He^{-sW}) || $')
# plt.title(f'Random Hamitlonian, seed={seed}')
plt.title(f'TFIM, n={nqubits} h={h}')

The above results show that the order of the magnetic expansion has no clear relation to the diagonalization. We then consider 2 factors:
1. Degeneracy
2. Ascending order

Now we sort the diagonal entried of the operators by ascending order and see if there is improvements.

In [ ]:
sorted_B_ops = [np.diag(np.sort(np.diag(matrix))) for matrix in B_ops]

In [ ]:
mode = DoubleBracketGeneratorType.single_commutator
scheduling = DoubleBracketScheduling.grid_search
off_diag_norm_orders = []
step_orders = []
for order in range(highest_order_test+1):
    dbi = DoubleBracketIteration(deepcopy(H),mode=mode, scheduling=scheduling)
    off_diag_norm = [dbi.off_diagonal_norm]
    steps = [0]
    for _ in range(NSTEPS):
        step = dbi.choose_step(d=sorted_B_ops[order])
        dbi(step=step, d=sorted_B_ops[order])
        off_diag_norm.append(dbi.off_diagonal_norm)
        steps.append(step)
    off_diag_norm_orders.append(off_diag_norm)
    step_orders.append(steps) 

In [ ]:
for order in range(highest_order_test+1):
    plt.plot(off_diag_norm_orders[order], label=f'order={order}')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel(r'$|| \sigma(e^{sW}He^{-sW}) || $')
# plt.title(f'Random Hamitlonian, seed={seed}')
plt.title(f'TFIM, n={nqubits} h={h} with sorted D')

We make the following observations:
1. The degree of diagonalization achieved by these fixed operators is highly dependent on the scheduling method used.
2. There exist scheduling methods such that all of the operators lead to diagonalization of the Hamiltonian.
3. Despite having more degeneracy, the unsorted $B_i=1$ generally gives the most diagonalization.
4. After the sort, the higher order $B$ fields (also less degeneracy) leads to faster initial diagonalization.

## Ising model

$D=\sum B_iZ_i + \sum J_{ij}Z_iZ_j$

In [ ]:
# function to generate operator strings
def generate_str_with_n_ops(nqubits, num_Z, pauli_name):
    op_str = []
    for num in range(num_Z):
        op_str += [''.join(pauli_name if i in positions else 'I' for i in range(nqubits)) 
            for positions in combinations(range(nqubits), num+1)]
    return op_str
# test
op_names = generate_str_with_n_ops(nqubits,2,"Z")
len(op_names)== nqubits+nqubits*(nqubits-1)/2

In [ ]:
# generate operators
ising_ops = [SymbolicHamiltonian(str_to_symbolic(op_name)).dense.matrix for op_name in op_names]
# dictionary
op_dict = dict(zip(str_names, ising_ops))


In [ ]:
# generate coefficients for the Ising model
def ising_coef_of_order(nqubits, num_Z, order):
    ising_coef = []
    for num in range(num_Z):
        for i in combinations(range(1,nqubits+1), num+1):
            ising_coef.append(math.prod(element**order for element in i))
    return np.array(ising_coef)

In [ ]:
linear_coef = ising_coef_of_order(nqubits, 2, 1)
linear_op = np.sum(linear_coef.reshape(-1,1,1)*ising_ops,0)

In [ ]:
visualize_matrix(linear_op)

In [ ]:
# visualize degeneracy
plt.plot(np.sort(np.diag(linear_op)))